# Crop Doctor: Notebook 01 - Dataset Audit & Distribution Check
**Project:** Offline Crop Disease/Pest Detection and Advisory (Arm Bharat AI-SoC Challenge 2026-27)

This notebook verifies:
1. Google Drive data mount and directory structure.
2. Image counts per class across all 25 target classes.
3. Class balance distribution.
4. Visual inspection grid (sample leaves per crop).

In [ ]:
# 1. Mount Google Drive (Reads user resized dataset)
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone repository to access class mapping and utilities
!git clone https://github.com/varun-gajula/crop-doctor-offline.git /content/crop-doctor-offline
%cd /content/crop-doctor-offline

In [ ]:
import os
import yaml
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Load target class mapping
with open('data/class_map.yaml', 'r', encoding='utf-8') as f:
    class_data = yaml.safe_load(f)

classes = class_data['classes']
print(f"Loaded {len(classes)} target classes from class_map.yaml")

In [ ]:
# Dataset Path on Google Drive
DATA_DIR = Path('/content/drive/MyDrive/crop-doctor/data_raw')

print(f"Inspecting dataset root: {DATA_DIR}")
if not DATA_DIR.exists():
    print("WARNING: Drive path not found. Please upload resized images to MyDrive/crop-doctor/data_raw/")
else:
    print("Google Drive directory connected successfully!")

In [ ]:
# Count images per class
summary = []
valid_exts = {'.jpg', '.jpeg', '.png', '.webp'}

for idx, info in classes.items():
    cid = info['class_id']
    crop = info['crop']
    name_en = info['name_en']
    name_te = info['name_te']
    
    # Search folder matches in DATA_DIR
    count = 0
    if DATA_DIR.exists():
        class_folder = DATA_DIR / cid
        if class_folder.exists():
            count = len([f for f in class_folder.rglob('*') if f.suffix.lower() in valid_exts])
    
    summary.append({
        'Index': idx,
        'Crop': crop.capitalize(),
        'Class ID': cid,
        'English Name': name_en,
        'Telugu Name': name_te,
        'Image Count': count,
        'Status': 'OK' if count >= 150 else ('LOW' if count > 0 else 'MISSING')
    })

df = pd.DataFrame(summary)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(df[['Index', 'Crop', 'English Name', 'Image Count', 'Status']].to_string(index=False))

total_images = df['Image Count'].sum()
print(f"\nTotal Verified Images: {total_images}")

In [ ]:
# Plot Class Distribution
if df['Image Count'].sum() > 0:
    plt.figure(figsize=(14, 6))
    plt.bar(df['Class ID'], df['Image Count'], color='teal')
    plt.xticks(rotation=90, fontsize=9)
    plt.ylabel('Image Count')
    plt.title('Crop Doctor - 25 Class Distribution')
    plt.tight_layout()
    plt.show()
else:
    print("Once datasets are uploaded to Google Drive, rerun this cell to view the distribution bar chart.")

In [ ]:
# Visual Inspection Grid: Render 1 sample per class
if DATA_DIR.exists() and total_images > 0:
    fig, axes = plt.subplots(5, 5, figsize=(15, 15))
    axes = axes.flatten()
    
    for i, info in enumerate(classes.values()):
        cid = info['class_id']
        folder = DATA_DIR / cid
        img_files = list(folder.glob('*.jpg')) if folder.exists() else []
        
        ax = axes[i]
        if img_files:
            img = Image.open(img_files[0])
            ax.imshow(img)
            ax.set_title(info['name_en'], fontsize=8)
        else:
            ax.text(0.5, 0.5, 'Missing', ha='center', va='center')
            ax.set_title(info['name_en'], fontsize=8)
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()